# Evaluate RSSM World Model on MiniGrid

Load a trained RSSM checkpoint and visualize imagined rollouts.

## 1. Setup

In [ ]:
!pip install gymnasium minigrid matplotlib pyyaml Pillow tqdm -q

import sys
!git clone https://github.com/mihalko711/tbank_intro_problem.git
sys.path.insert(0, 'tbank_intro_problem')
%cd tbank_intro_problem

## 2. Imports, config, environment

In [ ]:
import os
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt

from src import (
    RSSMWorldModel, collect_episode,
    get_env_properties, make_minigrid_env,
)

CHECKPOINT_PATH = 'checkpoints/MiniGrid-Empty-16x16-v0_default_final.pth'

with open('configs/minigrid_default.yml') as f:
    config = yaml.safe_load(f)
rssm_cfg = config['rssm']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

env = make_minigrid_env(config['environment_name'], seed=config['seed'])
obs_shape, action_size = get_env_properties(env)
print(f'Obs shape: {obs_shape}, action size: {action_size}')

## 3. Load trained model

In [ ]:
rssm = RSSMWorldModel(obs_shape, action_size, rssm_cfg, device)
rssm.load_checkpoint(CHECKPOINT_PATH)
print(f'Checkpoint loaded: {CHECKPOINT_PATH}')
print(f'Total gradient steps: {rssm.total_gradient_steps}')

## 4. Collect episodes into buffer

In [ ]:
print('Collecting 5 episodes...')
for _ in range(5):
    collect_episode(env, rssm, rssm.buffer)
print(f'Buffer size: {len(rssm.buffer)}')
env.close()

## 5. Visualize imagined rollouts

In [ ]:
@torch.no_grad()
def visualize_rollout(rssm, buffer, device, action_size, horizon=15, save_path=None):
    data = buffer.sample(1, horizon + 1)
    real_obs = data['observations'][0, 1:]

    obs_0 = data['observations'][0, 0].to(device)
    recurrent = torch.zeros(1, rssm.recurrent_size, device=device)
    latent = torch.zeros(1, rssm.latent_size, device=device)
    action = torch.zeros(1, action_size, device=device)

    recurrent, latent = rssm.encode_step(
        recurrent, latent, action, obs_0.cpu().numpy()
    )

    random_action_fn = lambda s: torch.nn.functional.one_hot(
        torch.randint(0, action_size, (1,), device=device), action_size
    ).float()

    imagined = rssm.rollout_prior(recurrent, latent, random_action_fn, horizon)
    prior_decoded = rssm.decoder(imagined.view(-1, rssm.full_state_size))
    prior_decoded = prior_decoded.view(horizon, *obs_shape)

    # ── Posterior reconstruction ──
    post_decoded = []
    rec, lat = recurrent.clone(), latent.clone()
    for t in range(horizon):
        full = torch.cat((rec, lat), -1)
        post_decoded.append(rssm.decoder(full).squeeze(0))
        if t < horizon - 1:
            enc_obs = rssm.encoder(real_obs[t].to(device).unsqueeze(0))
            rec = rssm.recurrent_model(rec, lat, data['actions'][0, t].unsqueeze(0).to(device))
            lat, _ = rssm.posterior_net(torch.cat((rec, enc_obs), -1))
    post_decoded = torch.stack(post_decoded)

    fig, axes = plt.subplots(3, horizon, figsize=(horizon * 2, 6))
    for t in range(horizon):
        axes[0, t].imshow(np.transpose(real_obs[t].cpu().numpy(), (1, 2, 0)))
        axes[0, t].axis('off')
        if t == 0:
            axes[0, t].set_title('Real', fontsize=10)

        axes[1, t].imshow(np.transpose(post_decoded[t].cpu().numpy().clip(0, 1), (1, 2, 0)))
        axes[1, t].axis('off')
        if t == 0:
            axes[1, t].set_title('Posterior', fontsize=10)

        axes[2, t].imshow(np.transpose(prior_decoded[t].cpu().numpy().clip(0, 1), (1, 2, 0)))
        axes[2, t].axis('off')
        if t == 0:
            axes[2, t].set_title('Imagined (prior)', fontsize=10)

    plt.suptitle(f'Real vs Posterior vs Imagined (horizon={horizon})')
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight', dpi=120)
    plt.show()


os.makedirs('visualizations', exist_ok=True)
for i in range(5):
    visualize_rollout(
        rssm, rssm.buffer, device, action_size, horizon=15,
        save_path=f'visualizations/eval_rollout_{i:02d}.png',
    )

print('Done.')